# Model Training Notebook
This notebook provides a complete workflow for data mining and machine learning model training.

## 1. Import Libraries

In [7]:
import sys
print(sys.executable)

c:\Users\lythu\AppData\Local\Programs\Python\Python313\python.exe


In [2]:
from sklearn.svm import LinearSVC
import numpy as np
import pandas as pd

In [3]:
data = pd.read_csv('CSV/train.csv')  
data_test = pd.concat([pd.read_csv('CSV/public_test.csv'), pd.read_csv('CSV/private_test.csv')])
data_test.head()

,id,title,venue,year,authors,doi
0,979,Explainable Planning Using Answer Set Programm...,kr,2020,"Van Nguyen, Stylianos Loukas Vasileiou, Tran C...",10.24963/kr.2020/66
1,1106,SOGrounder: Modelling and Solving Second-Order...,kr,2018,NaN,https://www.semanticscholar.org/paper/6968fcf3...
2,1894,Refinement for Structured Concurrent Programs.,cav,2020,NaN,https://www.semanticscholar.org/paper/23da9bfd...
3,1718,Randomized Synthesis for Diversity and Cost Co...,cav,2022,"Andreas Gittis, Eric Vin, Daniel J. Fremont",https://www.semanticscholar.org/paper/e23fc4b5...
4,1989,Inferring Inductive Invariants from Phase Stru...,cav,2019,"Yotam M. Y. Feldman, James R. Wilcox, Sharon S...",10.1007/978-3-030-25543-5_23


In [4]:
#Find the data types of each column
data.dtypes

id         int64
title        str
venue        str
year       int64
authors      str
doi          str
Label      int64
dtype: object

## 2. Clean Data 


In [5]:
#Check for missing values
data.isnull().sum()

id           0
title        0
venue        0
year         0
authors    192
doi          0
Label        0
dtype: int64

In [6]:
#Replace missing values of the authors column with unknown
columns_to_fill = ['authors']  # Specify which columns to fill
for col in columns_to_fill:
    if col in data.columns:
        data[col] = data[col].fillna('unknown')

In [7]:
#Recheck for missing values
data.isnull().sum()

id         0
title      0
venue      0
year       0
authors    0
doi        0
Label      0
dtype: int64

In [8]:
#Check if there is non ASCII text in the title and abstract columns
column_to_check = ['title']  # Specify which columns to check
for col in column_to_check:
    if col in data.columns:
        non_ASCII = data[~data[col].apply(lambda x: isinstance(x, str) and all(ord(c) < 128 for c in x))]
        print(f"Non-ASCII entries in column '{col}':")

Non-ASCII entries in column 'title':


## 3. TRAIN THE MODEL


In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# =========================
# Load datasets
# =========================



# =========================
# Prepare data
# =========================

X_train = data["title"].astype(str)
y_train = data["Label"]

X_test = data_test["title"].astype(str)

# =========================
# Create model pipeline
# =========================

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("logistic_regression", LogisticRegression())
])

# =========================
# Train model
# =========================

model.fit(X_train, y_train)

# =========================
# Predict
# =========================

predictions = model.predict(X_test)

# Convert predictions into labels 1-5
predictions = predictions.round().clip(1, 5)

# =========================
# Show predictions
# =========================

# Create a DataFrame with predictions
results_df = pd.DataFrame({
    'id': data_test['id'],
    'Label': predictions.astype(int)
})

results_df.to_csv('predictions.csv', index=False)

# Display first 10 predictions
print(results_df.head(10))
print(f"\nPredictions saved to 'predictions.csv' ({len(results_df)} rows)")

     id  Label
0   979      5
1  1106      2
2  1894      1
3  1718      1
4  1989      1
5  2010      1
6  2356      1
7  1179      1
8  3001      1
9   796      1

Predictions saved to 'predictions.csv' (596 rows)


In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

# =========================
# Load datasets
# =========================

# Example:
# data = pd.read_csv("train.csv")
# data_test = pd.read_csv("test.csv")

# =========================
# Prepare data
# =========================

X_train = data["title"].astype(str).tolist()
y_train = data["Label"]

X_test = data_test["title"].astype(str).tolist()

# =========================
# Load Sentence Transformer
# =========================

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

# =========================
# Convert titles to vectors
# =========================

X_train_embeddings = embedder.encode(
    X_train,
    show_progress_bar=True
)

X_test_embeddings = embedder.encode(
    X_test,
    show_progress_bar=True
)

# =========================
# Train classifier
# =========================

model = LogisticRegression(max_iter=1000)

model.fit(X_train_embeddings, y_train)

# =========================
# Predict
# =========================

predictions = model.predict(X_test_embeddings)

# Ensure labels stay between 1-5
predictions = predictions.clip(1, 5)

# =========================
# Save predictions
# =========================

results_df = pd.DataFrame({
    "id": data_test["id"],
    "Label": predictions.astype(int)
})

results_df.to_csv("predictions.csv", index=False)

# =========================
# Show results
# =========================

print(results_df.head(10))

print(f"\nPredictions saved to 'predictions.csv' ({len(results_df)} rows)")



c:\Users\lythu\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lythu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9098.85it/s]


AssertionError: Torch not compiled with CUDA enabled